In [1]:
import json
import os
import pickle

import numpy as np
# import pandas as pd
import torch
from B0_MODEL import UNET
from B1_UTILS import (identify_chain, load_checkpoint, map_normalizing,
                      map_writing, predict_location, smooth_location)
from B2_UTILS_modeling import refine_mass_centers, write_pdb  # track_5_3,

In [2]:
# ====================== Hyperparameters etc. ========================
MAIN_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST_with_blank'
MODEL_TYPES = ['secondary_strctures', 'key_atoms', 'residue_types']
MAP_DIRS_TYPES = [['map_sample'],
                  ['map_sample', 'secondary_strctures_pred_prob'],
                  [
                      'map_sample', 'secondary_strctures_pred_prob',
                      'key_atoms_pred_prob'
                  ]]
# IN_CHANNELS = [1, 6, 10]

data_path = os.path.join(MAIN_PATH, 'train_val_data')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:

def initial_setting(model_index, data_path, device):
    """
    This function performs the initial setup before training process:
    1. Read the class weights of the UNET model,
    2. Read the saved epoch number,
    3. Generate paths of file that will be loaded to UNET model.
    """

    model_type = MODEL_TYPES[model_index]
    # Load the class weights of all model types
    weight_path = os.path.join(data_path, 'class_weight_for_training.txt')
    try:
        # Load the weight for training
        with open(weight_path, "r") as file:
            weight_all = json.load(file)

        if model_type not in weight_all.keys():
            print(f"Weight for {model_type} is not found.")
            exit()
        else:
            c_w = weight_all[model_type]
            class_weights = torch.FloatTensor(c_w).to(device)

            # Calculate the number of output channels based on the number of tags in model_type
            out_channels = len(c_w)
            in_channels = 1
            if model_index > 0:
                for ii in range(model_index):
                    in_channels = in_channels + len(
                        weight_all[MODEL_TYPES[ii]])

    except FileNotFoundError:
        print("Weight file does not exist.")
        exit()

    # Load the load_epoch_num data or initialize it if the file doesn't exist
    load_epoch_num_path = os.path.join(data_path, "epoch_num_path.txt")

    if not os.path.exists(load_epoch_num_path):
        load_epoch_num_all = {model_type: 0}
    else:
        with open(load_epoch_num_path, "r") as file:
            load_epoch_num_all = json.load(file)
            if model_type not in load_epoch_num_all.keys():
                load_epoch_num_all[model_type] = 0
    load_epoch_num = load_epoch_num_all[model_type]

    # Save the updated load_epoch_num_all data
    with open(load_epoch_num_path, "w") as file:
        json.dump(load_epoch_num_all, file)

    load_model_path = os.path.join(
        data_path, 'checkpoint', model_type,
        f'{model_type}.epoch{load_epoch_num}.pth.tar')

    return class_weights, in_channels, out_channels, load_epoch_num, load_model_path


In [4]:
def output(MAP_PATH, out_channels, part_possibility, model_data, model_type,
           zoom_factors):

    folder_path = os.path.dirname(MAP_PATH)
    map_name = os.path.basename(MAP_PATH)

    for channel_num in range(1, out_channels):
        print(f"\n==> Start map writing of {model_type}: ")

        possibility = part_possibility[channel_num, :, :, :]
        out_map_file = f"{map_name.split('.mrc')[0]}_{model_type}_{channel_num}_possibility.mrc"
        out_map = os.path.join(folder_path, "possibility", out_map_file)
        map_writing(MAP_PATH, possibility, out_map, zoom_factors)

        label_data = np.where(model_data == channel_num, 1, 0)
        out_map_file = f"{map_name.split('.mrc')[0]}_{model_type}_{channel_num}.mrc"
        out_map = os.path.join(folder_path, "unrefined", out_map_file)
        map_writing(MAP_PATH, label_data, out_map, zoom_factors, is_int=True)

        # smooth_loc = smooth_location(label_data, channel_num)
        # out_map_file = f"{map_name.split('.mrc')[0]}_{model_type}_{channel_num}_refined.mrc"
        # out_map = os.path.join(folder_path, "refined", out_map_file)
        # map_writing(MAP_PATH, smooth_loc, out_map, zoom_factors, is_int=True)

In [5]:
RAW_DATA_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw'
MAP_PATH = os.path.join(RAW_DATA_PATH, 'EMD-0128_re_3.8', 'emd_0128.map')

# Create sub-folders
folder_path = os.path.dirname(MAP_PATH)
created_folders = ["possibility", "unrefined", "refined", "chains", "temp_in_chain"]
for created_folder in created_folders:
    os.makedirs(os.path.join(folder_path, created_folder), exist_ok=True)

map_data, zoom_factors = map_normalizing(MAP_PATH)


In [6]:
model_index = 0
model_type = MODEL_TYPES[model_index]
class_weights, in_channels, out_channels, load_epoch_num, load_model_path = (
    initial_setting(model_index, data_path, device=DEVICE)
)
print(f"\n==> Start {model_type} prediction: ")
model = UNET(in_channels, out_channels).to(DEVICE)
load_checkpoint(load_model_path, model, map_location=DEVICE)
map_data_ = np.expand_dims(map_data, axis=0)


==> Start secondary_strctures prediction: 


In [7]:
# Iterate over 3D cubes of the map data
# Seprate the map into 64*64*64 cubes but overlaps with 7 for every dimemsion
cube_size = 50
pad_size = 7

map_shape = map_data_.shape

cube_nums = [map_shape[i] // cube_size + 1 for i in range(1, len(map_shape))]
num_cubes = cube_nums[0] * cube_nums[1] * cube_nums[2]
larger_shape = [cube_size * cube_nums[i] + pad_size * 2 for i in range(len(cube_nums))]
larger_shape.insert(0, map_shape[0])

reshaped_map_data = np.zeros(tuple(larger_shape), np.float32)
reshaped_map_data[:, pad_size:pad_size+map_shape[1], pad_size:pad_size+map_shape[2], pad_size:pad_size+map_shape[3]] = map_data_

reshaped_map_data = torch.from_numpy(reshaped_map_data)
model_larger_shape = larger_shape[-3:]
model_larger_shape = np.insert(model_larger_shape, 0, out_channels)
reshaped_model = torch.zeros(tuple(model_larger_shape))

In [8]:
device = DEVICE

model.eval()
# Iterate over 3D cubes of the map data
with torch.no_grad():
    num_completed = 0

    for zz in range(cube_nums[0]):
        zstrt = cube_size * zz
        for yy in range(cube_nums[1]):
            ystrt = cube_size * yy
            for xx in range(cube_nums[2]):
                # Extract a 3D cube from the map data
                xstrt = cube_size * xx
                cube = reshaped_map_data[None, :, zstrt:64+zstrt, ystrt:64+ystrt, xstrt:64+xstrt]

                # Move the cube to the specified device and perform prediction
                cube = cube.to(device)
                pred_data = torch.softmax(model(cube), 1)

                # Write predicted data to reshaped_model
                reshaped_model[:, 7+zstrt:57+zstrt, 7+ystrt:57+ystrt, 7+xstrt:57+xstrt] = pred_data[0, :, 7:57, 7:57, 7:57]

                # Update progress
                num_completed += 1
                print(
                    f"{num_completed}/{num_cubes} of prediction is done.",
                    end='\r')

# Extract model data
part_possibility = reshaped_model[:, pad_size:pad_size+map_shape[1], pad_size:pad_size+map_shape[2], pad_size:pad_size+map_shape[3]].cpu().numpy()

# return part_possibility

In [ ]:
# ### TEST - generate predicted cubes


# TEST_PATH = "/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST_with_blank/train_val_data/val/map_sample"
# map_names = sorted(os.listdir(TEST_PATH))

# temp_sample_path = "/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST_with_blank/ready_to_train_and_val"

# device = DEVICE

# model.eval()
# # Iterate over 3D cubes of the map data
# with torch.no_grad():
#     num_completed = 0

#     for map_name in map_names:
#         map_path = os.path.join(TEST_PATH, map_name)
#         map_part = np.load(map_path)
#         cube1 = map_part[None, None, :, :, :]
#         cube = torch.tensor(cube1)

#         # Move the cube to the specified device and perform prediction
#         cube = cube.to(device)
#         pred_data = torch.softmax(model(cube), 1)

#         # Write predicted data to reshaped_model
#         writen = pred_data[0, :, :, :, :]
#         writen_1 = writen.cpu().numpy()
        
        
#         file_name = f"{model_type}_pred_prob.{map_name.split('.')[-2]}.npy"
#         file_path = os.path.join(temp_sample_path, f'{model_type}_pred_prob-test',
#                                  file_name)
        
#         np.save(file_path, writen_1)

#         # Update progress
#         num_completed += 1
#         print(
#             f"{num_completed}/{len(map_names)} of prediction is done.",
#             end='\r')

# # # Extract model data
# # part_possibility = reshaped_model[:, pad_size:pad_size+map_shape[1], pad_size:pad_size+map_shape[2], pad_size:pad_size+map_shape[3]].cpu().numpy()

# # # return part_possibility

In [14]:
# import splitfolders
# import shutil
# def split(temp_sample_path, sample_path):
#     # split data into training and testing dataset
#     os.makedirs(sample_path, exist_ok=True)
#     splitfolders.ratio(
#         input=temp_sample_path,
#         output=sample_path,
#         seed=44,
#         ratio=(0.8, 0.2),
#         group_prefix=None,
#         move=True,
#     )
#     shutil.rmtree(temp_sample_path)

# temp_sample_path = os.path.join(MAIN_PATH, "ready_to_train_and_val")
# sample_path = os.path.join(MAIN_PATH, "train_val_data")
# split(temp_sample_path, sample_path)

In [9]:
model_index = 1
model_type = MODEL_TYPES[model_index]
class_weights, in_channels, out_channels, load_epoch_num, load_model_path = initial_setting(
    model_index, data_path, device=DEVICE)
print(f"\n==> Start {model_type} prediction: ")
model = UNET(in_channels, out_channels).to(DEVICE)
load_checkpoint(load_model_path, model, map_location=DEVICE)
# map_data = np.expand_dims(map_data, axis=0)
map_data_ = np.concatenate((map_data_, part_possibility), axis=0)
map_data_.shape



==> Start key_atoms prediction: 


(6, 338, 338, 338)

In [10]:
map_shape = map_data_.shape

cube_size = 50
pad_size = 7

cube_nums = [map_shape[i] // cube_size + 1 for i in range(1, len(map_shape))]
num_cubes = cube_nums[0] * cube_nums[1] * cube_nums[2]
larger_shape = [cube_size * cube_nums[i] + pad_size * 2 for i in range(len(cube_nums))]
larger_shape.insert(0, map_shape[0])

reshaped_map_data = np.zeros(tuple(larger_shape), np.float32)
reshaped_map_data[:, pad_size:pad_size+map_shape[1], pad_size:pad_size+map_shape[2], pad_size:pad_size+map_shape[3]] = map_data_
reshaped_map_data = torch.from_numpy(reshaped_map_data)

model_larger_shape = larger_shape[-3:]
model_larger_shape = np.insert(model_larger_shape, 0, out_channels)
reshaped_model = torch.zeros(tuple(model_larger_shape))

In [11]:
device = DEVICE

model.eval()
# Iterate over 3D cubes of the map data
with torch.no_grad():
    num_completed = 0

    for zz in range(cube_nums[0]):
        zstrt = cube_size * zz
        for yy in range(cube_nums[1]):
            ystrt = cube_size * yy
            for xx in range(cube_nums[2]):
                # Extract a 3D cube from the map data
                xstrt = cube_size * xx
                cube = reshaped_map_data[None, :, zstrt:64+zstrt, ystrt:64+ystrt, xstrt:64+xstrt]

                # Move the cube to the specified device and perform prediction
                cube = cube.to(device)
                pred_data = torch.softmax(model(cube), 1)

                # Write predicted data to reshaped_model
                reshaped_model[:, 7+zstrt:57+zstrt, 7+ystrt:57+ystrt, 7+xstrt:57+xstrt] = pred_data[0, :, 7:57, 7:57, 7:57]

                # Update progress
                num_completed += 1
                print(
                    f"{num_completed}/{num_cubes} of prediction is done.",
                    end='\r')

# Extract model data
part_possibility = reshaped_model[:, pad_size:pad_size+map_shape[1], pad_size:pad_size+map_shape[2], pad_size:pad_size+map_shape[3]].cpu().numpy()

# return part_possibility

In [12]:
model_data = np.argmax(part_possibility, 0)
output(MAP_PATH, out_channels, part_possibility, model_data, model_type, zoom_factors)


==> Start map writing of key_atoms: 
Writing map: emd_0128.map_key_atoms_1_possibility.mrc
Writing map: emd_0128.map_key_atoms_1.mrc

==> Start map writing of key_atoms: 
Writing map: emd_0128.map_key_atoms_2_possibility.mrc
Writing map: emd_0128.map_key_atoms_2.mrc

==> Start map writing of key_atoms: 
Writing map: emd_0128.map_key_atoms_3_possibility.mrc
Writing map: emd_0128.map_key_atoms_3.mrc


In [10]:
y1 = part_possibility
y10 = np.max(y1[0]) #0.2
y11 = np.max(y1[1]) #0.21
y12 = np.max(y1[2]) #0.198
y13 = np.max(y1[3]) #0.20
y14 = np.max(y1[4]) #0.18
print(y10, y11, y12, y13, y14)

1.0 0.9999962 0.99999976 0.99999833 0.9995995


In [9]:
model_index = 2
model_type = MODEL_TYPES[model_index]
class_weights, in_channels, out_channels, load_epoch_num, load_model_path = initial_setting(
    model_index, data_path, device=DEVICE)
print(f"\n==> Start {model_type} prediction: ")
model = UNET(in_channels, out_channels).to(DEVICE)
load_checkpoint(load_model_path, model, map_location=DEVICE)
map_data_ = np.concatenate((map_data_, part_possibility), axis=0)
map_data_.shape



==> Start residue_types prediction: 


(6, 200, 200, 200)

In [10]:
map_shape = map_data_.shape

cube_size = 50
pad_size = 7

cube_nums = [map_shape[i] // cube_size + 1 for i in range(1, len(map_shape))]
num_cubes = cube_nums[0] * cube_nums[1] * cube_nums[2]
larger_shape = [cube_size * cube_nums[i] + pad_size * 2 for i in range(len(cube_nums))]
larger_shape.insert(0, map_shape[0])

reshaped_map_data = np.zeros(tuple(larger_shape), np.float32)
reshaped_map_data[:, pad_size:pad_size+map_shape[1], pad_size:pad_size+map_shape[2], pad_size:pad_size+map_shape[3]] = map_data_
reshaped_map_data = torch.from_numpy(reshaped_map_data)

model_larger_shape = larger_shape[-3:]
model_larger_shape = np.insert(model_larger_shape, 0, out_channels)
reshaped_model = torch.zeros(tuple(model_larger_shape))

In [11]:
model.eval()
# Iterate over 3D cubes of the map data
with torch.no_grad():
    num_completed = 0

    for zz in range(cube_nums[0]):
        zstrt = cube_size * zz
        for yy in range(cube_nums[1]):
            ystrt = cube_size * yy
            for xx in range(cube_nums[2]):
                # Extract a 3D cube from the map data
                xstrt = cube_size * xx
                cube = reshaped_map_data[None, :, zstrt:64+zstrt, ystrt:64+ystrt, xstrt:64+xstrt]

                # Move the cube to the specified device and perform prediction
                cube = cube.to(device)
                # a = model(cube)
                # print(f'model shape is {a.shape}')

                pred_data = torch.softmax(model(cube), 1)

                # Write predicted data to reshaped_model

                reshaped_model[:, 7+zstrt:57+zstrt, 7+ystrt:57+ystrt, 7+xstrt:57+xstrt] = pred_data[0, :, 7:57, 7:57, 7:57]

                # Update progress
                num_completed += 1
                print(
                    f"{num_completed}/{num_cubes} of prediction is done.",
                    end='\r')

# Extract model data
part_possibility = reshaped_model[:, pad_size:pad_size+map_shape[1], pad_size:pad_size+map_shape[2], pad_size:pad_size+map_shape[3]].cpu().numpy()

# return part_possibility

RuntimeError: Given groups=1, weight of size [64, 10, 3, 3, 3], expected input[1, 6, 64, 64, 64] to have 10 channels, but got 6 channels instead

In [ ]:
model_data = np.argmax(part_possibility, 0)
output(out_channels, part_possibility, model_data, model_type, zoom_factors)

In [ ]:
output(out_channels, part_possibility, model_data, model_type, zoom_factors)